cd contest2
source venv/bin/activate
cd ../
export LD_LIBRARY_PATH=$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cudnn/lib:$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cublas/lib:$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cusolver/lib:$VIRTUAL_ENV/lib/python3.10/site-packages/nvidia/cusparse/lib
python3 -c "import tensorflow as tf; print('Доступные GPU:', tf.config.list_physical_devices('GPU'))"
jupyter notebook --no-browser

In [2]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = Path('.')
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
LABELS_PATH = DATA_DIR / 'labels.csv'
SAMPLE_SUBMISSION_PATH = DATA_DIR / 'sample_submission.csv'

IMG_SIZE = 256
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

I0000 00:00:1779883559.064341    1323 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
labels = pd.read_csv(LABELS_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

labels['filename'] = labels['id'].astype(str) + '.jpg'
labels['filepath'] = labels['filename'].apply(lambda name: str(TRAIN_DIR / name))

classes = sorted(labels['breed'].unique())
class_to_index = {breed: idx for idx, breed in enumerate(classes)}
labels['label'] = labels['breed'].map(class_to_index)


In [4]:
train_df, valid_df = train_test_split(
    labels,
    test_size=0.2,
    random_state=SEED,
    stratify=labels['breed'],
)

In [5]:
def decode_image(path, label=None):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = preprocess_input(image)
    if label is None:
        return image
    return image, tf.one_hot(label, depth=len(classes))

def make_dataset(df, training=False):
    paths = df['filepath'].values
    labels_array = df['label'].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels_array))
    if training:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
valid_ds = make_dataset(valid_df, training=False)

I0000 00:00:1779883588.646967    1323 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3582 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [6]:
augmentation = keras.Sequential(
    [
        layers.RandomFlip('horizontal', seed=SEED),
        layers.RandomRotation(0.08, seed=SEED),
        layers.RandomZoom(0.12, seed=SEED),
        layers.RandomTranslation(0.08, 0.08, seed=SEED),
    ],
    name='augmentation',
)

In [7]:
def build_model():
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = augmentation(inputs)

    base_model = MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
    )
    base_model.trainable = False

    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(len(classes), activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    return model, base_model

model, base_model = build_model()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')],
)

/tmp/ipykernel_1323/1619631259.py:5: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


In [8]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        'mobilenetv2_dog_breeds_best.keras',
        monitor='val_loss',
        save_best_only=True,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=2,
        min_lr=1e-6,
    ),
]

initial_epochs = 5
history_frozen = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=initial_epochs,
    callbacks=callbacks,
)

Epoch 1/5


I0000 00:00:1779883598.167504    1407 cuda_dnn.cc:461] Loaded cuDNN version 92200


256/256 ━━━━━━━━━━━━━━━━━━━━ 32s 97ms/step - accuracy: 0.4949 - loss: 2.2152 - top_5_accuracy: 0.7762 - val_accuracy: 0.7917 - val_loss: 0.8524 - val_top_5_accuracy: 0.9751 - learning_rate: 0.0010
Epoch 2/5
256/256 ━━━━━━━━━━━━━━━━━━━━ 24s 92ms/step - accuracy: 0.7439 - loss: 0.9177 - top_5_accuracy: 0.9594 - val_accuracy: 0.8117 - val_loss: 0.6370 - val_top_5_accuracy: 0.9804 - learning_rate: 0.0010
Epoch 3/5
256/256 ━━━━━━━━━━━━━━━━━━━━ 21s 81ms/step - accuracy: 0.7969 - loss: 0.7211 - top_5_accuracy: 0.9708 - val_accuracy: 0.8269 - val_loss: 0.5736 - val_top_5_accuracy: 0.9814 - learning_rate: 0.0010
Epoch 4/5
256/256 ━━━━━━━━━━━━━━━━━━━━ 24s 93ms/step - accuracy: 0.8179 - loss: 0.6100 - top_5_accuracy: 0.9803 - val_accuracy: 0.8210 - val_loss: 0.5560 - val_top_5_accuracy: 0.9824 - learning_rate: 0.0010
Epoch 5/5
256/256 ━━━━━━━━━━━━━━━━━━━━ 24s 93ms/step - accuracy: 0.8446 - loss: 0.5354 - top_5_accuracy: 0.9851 - val_accuracy: 0.8377 - val_loss: 0.5243 - val_top_5_accuracy: 0.9848

In [9]:
base_model.trainable = True

fine_tune_at = int(len(base_model.layers) * 0.8) // 2
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
for layer in base_model.layers[fine_tune_at:]:
    layer.trainable = True

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(
        monitor='val_loss', 
        patience=3,  
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')],
)

fine_tune_epochs = 6
total_epochs = initial_epochs + fine_tune_epochs

history_fine = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=total_epochs,
    initial_epoch=history_frozen.epoch[-1] + 1,
    callbacks=callbacks,
)

Epoch 6/11
256/256 ━━━━━━━━━━━━━━━━━━━━ 62s 173ms/step - accuracy: 0.7702 - loss: 0.7423 - top_5_accuracy: 0.9661 - val_accuracy: 0.7936 - val_loss: 0.6511 - val_top_5_accuracy: 0.9790 - learning_rate: 5.0000e-05
Epoch 7/11
256/256 ━━━━━━━━━━━━━━━━━━━━ 45s 177ms/step - accuracy: 0.8348 - loss: 0.5133 - top_5_accuracy: 0.9874 - val_accuracy: 0.8108 - val_loss: 0.6008 - val_top_5_accuracy: 0.9790 - learning_rate: 5.0000e-05
Epoch 8/11
256/256 ━━━━━━━━━━━━━━━━━━━━ 43s 166ms/step - accuracy: 0.8646 - loss: 0.4149 - top_5_accuracy: 0.9927 - val_accuracy: 0.7941 - val_loss: 0.6406 - val_top_5_accuracy: 0.9785 - learning_rate: 5.0000e-05
Epoch 9/11
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step - accuracy: 0.8950 - loss: 0.3429 - top_5_accuracy: 0.9948
Epoch 9: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-05.
256/256 ━━━━━━━━━━━━━━━━━━━━ 42s 165ms/step - accuracy: 0.8869 - loss: 0.3588 - top_5_accuracy: 0.9945 - val_accuracy: 0.7932 - val_loss: 0.6597 - val_top_5_accuracy: 0.973

In [12]:
RUN_FINAL_TRAINING = True

if RUN_FINAL_TRAINING:
    final_model, final_base_model = build_model()
    
    final_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')],
    )
    
    history_phase1 = final_model.fit(
        train_ds,  
        validation_data=valid_ds,  
        epochs=initial_epochs,
        verbose=1, 
        callbacks=[
            EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
        ]
    )
    
    final_base_model.trainable = True
    fine_tune_at = len(final_base_model.layers) // 2
    
    for layer in final_base_model.layers[:fine_tune_at]:
        layer.trainable = False
    
    final_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=5e-5),  
        loss='categorical_crossentropy',
        metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')],
    )
    
    history_phase2 = final_model.fit(
        train_ds,
        validation_data=valid_ds,
        epochs=fine_tune_epochs,
        verbose=1,
        callbacks=[
            EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
        ]
    )
    
    model = final_model

/tmp/ipykernel_1323/1619631259.py:5: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


Epoch 1/5
256/256 ━━━━━━━━━━━━━━━━━━━━ 25s 83ms/step - accuracy: 0.4873 - loss: 2.2309 - top_5_accuracy: 0.7703 - val_accuracy: 0.7878 - val_loss: 0.8668 - val_top_5_accuracy: 0.9800 - learning_rate: 0.0010
Epoch 2/5
256/256 ━━━━━━━━━━━━━━━━━━━━ 23s 91ms/step - accuracy: 0.7455 - loss: 0.9237 - top_5_accuracy: 0.9600 - val_accuracy: 0.8171 - val_loss: 0.6494 - val_top_5_accuracy: 0.9829 - learning_rate: 0.0010
Epoch 3/5
256/256 ━━━━━━━━━━━━━━━━━━━━ 24s 92ms/step - accuracy: 0.7933 - loss: 0.7203 - top_5_accuracy: 0.9698 - val_accuracy: 0.8254 - val_loss: 0.5884 - val_top_5_accuracy: 0.9819 - learning_rate: 0.0010
Epoch 4/5
256/256 ━━━━━━━━━━━━━━━━━━━━ 21s 80ms/step - accuracy: 0.8169 - loss: 0.6185 - top_5_accuracy: 0.9786 - val_accuracy: 0.8308 - val_loss: 0.5462 - val_top_5_accuracy: 0.9824 - learning_rate: 0.0010
Epoch 5/5
256/256 ━━━━━━━━━━━━━━━━━━━━ 23s 91ms/step - accuracy: 0.8375 - loss: 0.5315 - top_5_accuracy: 0.9854 - val_accuracy: 0.8200 - val_loss: 0.5501 - val_top_5_accura

In [13]:
test_paths = [str(TEST_DIR / f'{image_id}.jpg') for image_id in sample_submission['id']]
test_ds = tf.data.Dataset.from_tensor_slices(test_paths)
test_ds = test_ds.map(lambda path: decode_image(path), num_parallel_calls=AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

predictions = model.predict(test_ds, verbose=0)

submission = pd.DataFrame(predictions, columns=classes)
submission.insert(0, 'id', sample_submission['id'].values)
submission = submission[sample_submission.columns]
submission.to_csv('submission2.csv', index=False)